In [62]:
import requests
from bs4 import BeautifulSoup, Comment
import pandas as pd

In [24]:
#translating the scraping better for better scalability
url = 'https://www.pro-football-reference.com/teams/clt/2004.htm' 
response = requests.get(url)
print(response.status_code)

200


In [25]:
#Colts statistics page
text2 = BeautifulSoup(response.content, 'html.parser')

In [63]:
table_ids = [table.get("id") for table in text2.find_all("table") if table.get("id")]
print(table_ids)

['team_stats', 'games', 'team_conversions', 'passing', 'passing_post', 'rushing_and_receiving', 'rushing_and_receiving_post', 'defense', 'defense_post']


In [56]:
all_rows["passing"]

[<tr> <th aria-label="Rk" class="ranker poptip center" data-stat="ranker" data-tip="Rank" scope="col">Rk</th> <th aria-label="Player" class="poptip sort_default_asc center" data-stat="name_display" scope="col">Player</th> <th aria-label="Age" class="poptip center" data-stat="age" data-tip="As of Dec. 31 of the season in question." scope="col">Age</th> <th aria-label="Pos" class="poptip center" data-stat="pos" data-tip="Position" scope="col">Pos</th> <th aria-label="G" class="poptip center" data-stat="games" data-tip="Games played" scope="col">G</th> <th aria-label="GS" class="poptip center" data-stat="games_started" data-tip="Games started as an offensive or defensive player" scope="col">GS</th> <th aria-label="QBrec" class="poptip center" data-stat="qb_rec" data-tip="Team record in games started by this QB (regular season)" scope="col">QBrec</th> <th aria-label="Cmp" class="poptip center" data-stat="pass_cmp" data-tip="Passes completed" scope="col">Cmp</th> <th aria-label="Att" class=

In [27]:
comments = text2.find_all(string=lambda text: isinstance(text, Comment))
tables = []
for c in comments:
    comment_soup = BeautifulSoup(c, "html.parser")
    table = comment_soup.find("table")
    if table:
        tables.append(table)

print(f"Found {len(tables)} hidden tables")
print([t.get("id") for t in tables])  # print their ids

Found 6 hidden tables
['returns', 'kicking', 'punting', 'scoring', 'team_td_log', 'opp_td_log']


In [30]:
#extract all rows and columns for each table
regular = ['passing', 'rushing_and_receiving', 'defense']
overall = ['team_stats', 'games', 'team_conversions']
hidden = tables
all_rows = dict()
all_cols = dict()
for id in table_ids:
    cur = text2.find('table', {'id': id})
    cur_rows= cur.find_all('tr')
    if (id in regular):
        next_id = table_ids[table_ids.index(id) + 1]
        post = text2.find('table', {'id': next_id})
        post_rows = post.find_all('tr')
        all_rows[id] = cur_rows + post_rows
    elif (id in overall):
        all_rows[id] = cur_rows
    else:
        continue
    columnT = cur.find_all('th')
    cols = []
    for title in columnT:
        header = title.get('aria-label')
        cols.append(header)
    all_cols[id] = cols
for table in tables:
    id = table.get("id")
    cur_rows= table.find_all('tr')
    all_rows[id] = cur_rows
        
    columnT = table.find_all('th')
    cols = []
    for title in columnT:
        header = title.get('aria-label')
        cols.append(header)
    all_cols[id] = cols
        
    
        
    

In [51]:
print(all_cols['team_td_log'])

['Rk', 'Date', ' ', 'Opp', 'Result', 'Quarter', 'Dist', 'Type', 'Detail', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


In [38]:
for id in all_cols:
    print(id)

team_stats
games
team_conversions
passing
rushing_and_receiving
defense
returns
kicking
punting
scoring
team_td_log
opp_td_log


In [20]:

def find_duplicates(input_list):
    seen = set()
    duplicates = dict()
    for i in range(len(input_list)):
        item = input_list[i]
        if item not in seen:
            seen.add(item)
        else:
            if item in duplicates:
                duplicates[item][0] += 1
                duplicates[item][1].append(i)
            else:
                duplicates[item] = [1,[i]]
    return duplicates

In [59]:
#modify and cleanse columns (remove duplicates)
def modify_cols(all_cols, id):
    cleaned = []
    cols = all_cols.get(id)
    if (id == 'team_stats'):
        cleaned = cols[12:(cols.index("Pts") + 1)]


            

    elif (id == 'games'):
        cleaned = cols[6:(cols.index("Sp. Tms") + 1)]
        
    elif (id == 'passing'):
        cleaned = cols[1:(cols.index("Awards") + 1)]
    elif (id == 'rushing_and_receiving'):
        cleaned = cols[7:(cols.index("Awards") + 1)]
    elif (id == 'team_conversions'):
        cleaned = cols[4:(cols.index("Red Zone Score Pct") + 1)]
    elif (id == 'table_pfr_team-year_game-logs_team-year-regular-season-game-log'
     or id == 'table_pfr_team-year_game-logs_team-year-regular-season-opponent-game-log'):
        cleaned = cols[13:(cols.index("ToP") + 1)]
    elif (id == 'returns'):
        cleaned = cols[5:(cols.index("Awards") + 1)]
    elif (id == 'kicking'):
        cleaned = cols[10:(cols.index("Awards") + 1)]
    elif (id == 'punting'):
        cleaned = cols[4:(cols.index("Awards") + 1)]
    elif (id == 'scoring' or id == 'defense'):
        cleaned = cols[8:(cols.index("Awards") + 1)]
    else:
        cleaned = cols[1:(cols.index("Detail") + 1)]
    
    duplicates = find_duplicates(cleaned)
    for item in duplicates:
        count = duplicates[item][0]
        all_duplicates = duplicates[item][1]
        cur_word = item
        for i in range(count):
            idx = all_duplicates[i]
            cur_word += 'R'
            cleaned[idx] = cur_word
    return cleaned

    
    



In [61]:
import os
#get all csvs
for id in all_rows:

    data = []
    rows = all_rows[id]
    cols = all_cols[id]
    for row in rows:
        cols = row.find_all(['td'])
        cols = [col.text.strip() for col in cols]
        data.append(cols)
    cleaned_col_names = modify_cols(all_cols, id)
    df = pd.DataFrame(data, columns=[cleaned_col_names])  # Replace with actual column names
    df = df.drop([0])
    df = df.reset_index(drop=True)
    folder_path = 'Colts statistics page'
    file_name = id + '1.csv'
    path = os.path.join(folder_path, file_name)
    df.to_csv(path, index=False)

In [64]:
#more detailed team and opp game logs
url2 = 'https://www.pro-football-reference.com/teams/clt/2004/gamelog/' 
response2 = requests.get(url2)
print(response2.status_code)

200


In [65]:
text3 = BeautifulSoup(response2.content, 'html.parser')
print(text3.prettify())

<!DOCTYPE html>
<html class="no-js" data-root="/home/pfr/deploy/www" data-version="klecko-" lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="ie=edge" http-equiv="x-ua-compatible"/>
  <meta content="width=device-width, initial-scale=1.0, maximum-scale=2.0" name="viewport">
   <link href="https://cdn.ssref.net/req/202508071" rel="dns-prefetch"/>
   <script>
    /* https://docs.osano.com/hc/en-us/articles/22469433444372-Google-Consent-Mode-v2  */
  window.dataLayer = window.dataLayer ||[];
      function gtag(){dataLayer.push(arguments);}
      gtag('consent','default',{
        'ad_storage':'denied',
        'analytics_storage':'denied',
        'ad_user_data':'denied',
        'ad_personalization':'denied',
        'personalization_storage':'denied',
        'functionality_storage':'granted',
        'security_storage':'granted',
        'wait_for_update': 500
      });
      gtag("set", "ads_data_redaction", true);
   </script>
   <script src="https://cmp.osano.com/16CGnCU

In [66]:
table_ids2 = [table.get("id") for table in text3.find_all("table") if table.get("id")]
print(table_ids2)


['table_pfr_team-year_game-logs_team-year-regular-season-game-log', 'table_pfr_team-year_game-logs_team-year-playoffs-game-log', 'table_pfr_team-year_game-logs_team-year-regular-season-opponent-game-log', 'table_pfr_team-year_game-logs_team-year-playoffs-opponent-game-log']


In [68]:
all_rows2 = dict()
all_cols2 = dict()
for id in table_ids2:
    cur = text3.find('table', {'id': id})
    cur_rows= cur.find_all('tr')
    if ('playoffs' not in id):
        next_id = table_ids2[table_ids2.index(id) + 1]
        post = text3.find('table', {'id': next_id})
        post_rows = post.find_all('tr')
        all_rows2[id] = cur_rows + post_rows
    else:
        continue
        
    columnT = cur.find_all('th')
    cols = []
    for title in columnT:
        header = title.get('aria-label')
        cols.append(header)
    all_cols2[id] = cols
        

In [69]:
all_rows.update(all_rows2)
all_cols.update(all_cols2)

In [70]:
import os
#get all csvs
for id in all_rows:

    data = []
    rows = all_rows[id]
    cols = all_cols[id]
    for row in rows:
        cols = row.find_all(['td'])
        cols = [col.text.strip() for col in cols]
        data.append(cols)
    cleaned_col_names = modify_cols(all_cols, id)
    df = pd.DataFrame(data, columns=[cleaned_col_names])  # Replace with actual column names
    df = df.drop([0])
    df = df.reset_index(drop=True)
    folder_path = 'Colts statistics page'
    file_name = id + '1.csv'
    path = os.path.join(folder_path, file_name)
    df.to_csv(path, index=False)

In [29]:
#advanced stats
url3 = 'https://www.pro-football-reference.com/years/2003/index.htm' 
response3 = requests.get(url3)
print(response3.status_code)
text4 = BeautifulSoup(response3.content, 'html.parser')

200


In [30]:
comments = text4.find_all(string=lambda text: isinstance(text, Comment))
tables = []
for c in comments:
    comment_soup = BeautifulSoup(c, "html.parser")
    table = comment_soup.find("table")
    if table:
        tables.append(table)

print(f"Found {len(tables)} hidden tables")
print([t.get("id") for t in tables])  # print their ids

Found 12 hidden tables
['playoff_results', 'afc_playoff_standings', 'nfc_playoff_standings', 'team_stats', 'passing', 'rushing', 'returns', 'kicking', 'punting', 'team_scoring', 'team_conversions', 'drives']


In [31]:
table_ids3 = [table.get("id") for table in tables]
table_ids3

['playoff_results',
 'afc_playoff_standings',
 'nfc_playoff_standings',
 'team_stats',
 'passing',
 'rushing',
 'returns',
 'kicking',
 'punting',
 'team_scoring',
 'team_conversions',
 'drives']

In [36]:
#extract all rows and columns for each table
all_rows3 = dict()
all_cols3 = dict()
for table in tables:
    id = table.get("id")
    cur_rows= table.find_all('tr')
    all_rows3[id] = cur_rows
        
    columnT = table.find_all('th')
    cols = []
    for title in columnT:
        header = title.get('aria-label')
        cols.append(header)
    all_cols3[id] = cols
        
    

In [37]:
all_rows3

{'playoff_results': [<tr>
  <th aria-label="Week" class="poptip sort_default_asc sorttable_nosort center" data-stat="week_num" data-tip="Week number in season" scope="col">Week</th>
  <th aria-label="Day" class="poptip sort_default_asc left" data-stat="game_day_of_week" scope="col">Day</th>
  <th aria-label="Date" class="poptip sort_default_asc center" data-stat="game_date" scope="col">Date</th>
  <th aria-label="Winner/tie" class="poptip sort_default_asc center" data-stat="winner" scope="col">Winner/tie</th>
  <th aria-label="" class="poptip sort_default_asc center" data-stat="game_location" scope="col"></th>
  <th aria-label="Loser/tie" class="poptip sort_default_asc center" data-stat="loser" scope="col">Loser/tie</th>
  <th aria-label="" class="poptip sort_default_asc center" data-stat="boxscore_word" scope="col"></th>
  <th aria-label="Pts" class="poptip center" data-stat="pts_win" data-tip="Points Scored by the winning team (first one listed)" scope="col">Pts</th>
  <th aria-label

In [44]:
print(table_ids3)

['playoff_results', 'afc_playoff_standings', 'nfc_playoff_standings', 'team_stats', 'passing', 'rushing', 'returns', 'kicking', 'punting', 'team_scoring', 'team_conversions', 'drives']


In [50]:
for id in all_cols3:
    print(all_cols3[id])

['Week', 'Day', 'Date', 'Winner/tie', '', 'Loser/tie', '', 'Pts', 'Pts', None, None, None, None, None, None, None, None, None, None, None]
['Tm', 'Wins', 'Losses', 'Ties', 'Position', 'Reason', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
['Tm', 'Wins', 'Losses', 'Ties', 'Position', 'Reason', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]
['', None, None, None, '', None, None, '', '', '', '', None, 'Rk', 'Tm', 'Games', 'Points For', 'Yds', ' Offensive Plays Run ', ' Yards Per Offensive Play ', 'Turnovers', 'Fumbles Lost', '1st Downs', 'Passes Completed', 'Pass Attempts', 'Passing Yds', 'Passing TD', 'Passes Intercepted', 'Net Yds/Pass Att', '1stD', 'Rushing Att', 'Rushing Yds', 'Rushing TD', 'Yds/Rushing Att', '1stD', 'Penalties', 'Penalty Yds', '1stPy', 'Sc%', 'TO%', 'EXP', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, 

In [71]:
#modify and cleanse columns (remove duplicates)
def modify_cols2(all_cols, id):
    cleaned = []
    cols = all_cols.get(id)
    if (id == 'playoff_results'):
        cleaned = cols[0:9]
    
    elif (id == 'AFC' or id == 'NFC'):
        cleaned = cols[1:(cols.index("DSRS") + 1)]

    elif (id == 'afc_playoff_standings' or id == 'nfc_playoff_standings'):
        cleaned = cols[1:(cols.index("Reason") + 1)]
        
    elif (id == 'team_stats'):
        cleaned = cols[13:(cols.index("EXP") + 1)]
    elif (id == 'passing' or id == 'rushing'):
        cleaned = cols[1:(cols.index("EXP") + 1)]
    elif (id == 'returns'):
        cleaned = cols[6:(cols.index("All-Purpose Yds") + 1)]
    elif (id == 'kicking'):
        cleaned = cols[10:(cols.index("Avg. Kickoff Yards") + 1)]
    elif (id == 'punting'):
        cleaned = cols[4:(cols.index("Punts Blocked") + 1)]
    elif (id == 'team_scoring'):
        cleaned = cols[1:(cols.index("Pts/G") + 1)]
    elif (id == 'team_conversions'):
        cleaned = cols[5:(cols.index("Red Zone Score Pct") + 1)]
    else:
        cleaned = cols[5:(cols.index("Pts") + 1)]
    
    duplicates = find_duplicates(cleaned)
    for item in duplicates:
        count = duplicates[item][0]
        all_duplicates = duplicates[item][1]
        cur_word = item
        for i in range(count):
            idx = all_duplicates[i]
            cur_word += 'R'
            cleaned[idx] = cur_word
    return cleaned



In [79]:
#for all advanced stats
for id in all_rows3:

    data = []
    rows = all_rows3[id]
    cols = all_cols3[id]
    for row in rows:
        cols = row.find_all(['td', 'th'])
        cols = [col.text.strip() for col in cols]
        data.append(cols)
    cleaned_col_names = modify_cols2(all_cols3, id)
    df = pd.DataFrame(data, columns=[cleaned_col_names])  # Replace with actual column names
    df = df.drop([0, 1])
    df.to_csv(id + '1.csv', index=False)

In [66]:
#afc and nfc standings
standings= [table.get("id") for table in text4.find_all("table") if table.get("id")]
print(standings)

['AFC', 'NFC']


In [68]:
for id in standings:
    table = text4.find('table', {'id': id})
    rows = table.find_all('tr')
    all_rows3[id] = rows
    columnT = table.find_all('th')
    cols = []
    for title in columnT:
        header = title.get('aria-label')
        cols.append(header)
    all_cols3[id] = cols
    

In [73]:
all_cols3["playoff_results"]

['Week',
 'Day',
 'Date',
 'Winner/tie',
 '',
 'Loser/tie',
 '',
 'Pts',
 'Pts',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [70]:
all_cols3["NFC"]

['Tm',
 'Wins',
 'Losses',
 'W-L%',
 'Points For',
 'Points Allowed',
 'Points Differential',
 'MoV',
 'SoS',
 'SRS',
 'OSRS',
 'DSRS',
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [74]:
all_rows3["AFC"]

[<tr>
 <th aria-label="Tm" class="poptip sort_default_asc show_partial_when_sorting left" data-stat="team" scope="col">Tm</th>
 <th aria-label="Wins" class="poptip center" data-stat="wins" data-tip="Games Won" scope="col">W</th>
 <th aria-label="Losses" class="poptip center" data-stat="losses" data-tip="Games Lost" scope="col">L</th>
 <th aria-label="W-L%" class="poptip hide_non_quals center" data-stat="win_loss_perc" data-tip="&lt;b&gt;Win-Loss Percentage of team&lt;/b&gt;&lt;br&gt;For coaches, minimum to qualify for leading is 50 games.&lt;br&gt;After 1972, ties are counted as half-wins and half-losses.&lt;br&gt;Prior, the league didn\'t count them as games in W-L% calculations." scope="col">W-L%</th>
 <th aria-label="Points For" class="poptip center" data-stat="points" data-tip="Points Scored by team" scope="col">PF</th>
 <th aria-label="Points Allowed" class="poptip center" data-stat="points_opp" data-tip="Points Scored by opposition" scope="col">PA</th>
 <th aria-label="Points Dif

In [1]:
import nfl_data_py as nfl

In [2]:
pbp = nfl.import_pbp_data(years=[2003])

2003 done.
Downcasting floats.


In [4]:
coltspbp= pbp[(pbp["posteam"] == 'IND') | (pbp["defteam"] == 'IND')]
coltspbp

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
1077,35.0,2003_01_IND_CLE,2003090702,CLE,IND,REG,1,IND,away,CLE,...,0.0,0.0,1.164213,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1078,57.0,2003_01_IND_CLE,2003090702,CLE,IND,REG,1,IND,away,CLE,...,0.0,0.0,1.010665,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1079,78.0,2003_01_IND_CLE,2003090702,CLE,IND,REG,1,IND,away,CLE,...,0.0,0.0,-0.771268,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1081,116.0,2003_01_IND_CLE,2003090702,CLE,IND,REG,1,IND,away,CLE,...,0.0,0.0,1.716351,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1082,145.0,2003_01_IND_CLE,2003090702,CLE,IND,REG,1,IND,away,CLE,...,0.0,0.0,-0.496155,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46606,4145.0,2003_20_IND_NE,2004011800,NE,IND,POST,20,IND,away,NE,...,0.0,1.0,-0.232258,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46607,4178.0,2003_20_IND_NE,2004011800,NE,IND,POST,20,IND,away,NE,...,0.0,1.0,-0.249339,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46608,4197.0,2003_20_IND_NE,2004011800,NE,IND,POST,20,IND,away,NE,...,1.0,1.0,-0.175322,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46609,4218.0,2003_20_IND_NE,2004011800,NE,IND,POST,20,NE,home,IND,...,0.0,1.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
coltspbp = coltspbp.fillna(0)
coltspbp

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
1077,35.0,2003_01_IND_CLE,2003090702,CLE,IND,REG,1,IND,away,CLE,...,0.0,0.0,1.164213,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1078,57.0,2003_01_IND_CLE,2003090702,CLE,IND,REG,1,IND,away,CLE,...,0.0,0.0,1.010665,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1079,78.0,2003_01_IND_CLE,2003090702,CLE,IND,REG,1,IND,away,CLE,...,0.0,0.0,-0.771268,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1081,116.0,2003_01_IND_CLE,2003090702,CLE,IND,REG,1,IND,away,CLE,...,0.0,0.0,1.716351,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1082,145.0,2003_01_IND_CLE,2003090702,CLE,IND,REG,1,IND,away,CLE,...,0.0,0.0,-0.496155,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46606,4145.0,2003_20_IND_NE,2004011800,NE,IND,POST,20,IND,away,NE,...,0.0,1.0,-0.232258,0.0,0.0,0.0,0.0,0.0,0.0,0.0
46607,4178.0,2003_20_IND_NE,2004011800,NE,IND,POST,20,IND,away,NE,...,0.0,1.0,-0.249339,0.0,0.0,0.0,0.0,0.0,0.0,0.0
46608,4197.0,2003_20_IND_NE,2004011800,NE,IND,POST,20,IND,away,NE,...,1.0,1.0,-0.175322,0.0,0.0,0.0,0.0,0.0,0.0,0.0
46609,4218.0,2003_20_IND_NE,2004011800,NE,IND,POST,20,NE,home,IND,...,0.0,1.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
import pandas as pd
mappings = pd.read_csv("nfl_pbp_column_categories_v3 (1).csv")
mappings

,column,category
0,play_id,Identifiers & Meta
1,game_id,Identifiers & Meta
2,old_game_id,Identifiers & Meta
3,home_team,Teams & Coaches
4,away_team,Teams & Coaches
...,...,...
366,xyac_mean_yardage,Advanced Models & Probabilities
367,xyac_median_yardage,Advanced Models & Probabilities
368,xyac_success,Advanced Models & Probabilities
369,xyac_fd,Advanced Models & Probabilities


In [10]:
categories = mappings.groupby("category").agg(list).reset_index()
categories

,category,column
0,Admin / Data Quality,[play_deleted]
1,Advanced Models & Probabilities,"[no_score_prob, opp_fg_prob, opp_safety_prob, ..."
2,Betting & Vegas,"[vegas_wpa, vegas_home_wpa, vegas_wp, vegas_ho..."
3,Drive Data,"[drive, fixed_drive, fixed_drive_result, drive..."
4,Environment & Stadium,"[stadium, weather, location, div_game, roof, s..."
5,Fantasy,"[fantasy_player_name, fantasy_player_id, fanta..."
6,Identifiers & Meta,"[play_id, game_id, old_game_id, season_type, w..."
7,Play Classification & Formation,"[play_type, shotgun, no_huddle, qb_dropback, q..."
8,Play Outcomes & Events,"[ydstogo, ydsnet, desc, yards_gained, first_do..."
9,Player Participation – Defense & ST,"[interception_player_id, interception_player_n..."


In [11]:
categories.iloc[1][0]

'Advanced Models & Probabilities'

In [29]:
for i in range(categories.shape[0]):
    if i == 0:
        continue
    category = categories.iloc[i][0]
    
    cols = categories.iloc[i][1]
    cur_df = coltspbp[cols]
    cur_df.to_csv(category + "1.csv", index=False)


In [17]:
#weekly stats for each player(offense)

In [16]:
probabilities = pd.read_csv("Advanced Models & Probabilities1.csv")
cols = list(probabilities.columns)
cols

['no_score_prob',
 'opp_fg_prob',
 'opp_safety_prob',
 'opp_td_prob',
 'fg_prob',
 'safety_prob',
 'td_prob',
 'extra_point_prob',
 'two_point_conversion_prob',
 'ep',
 'epa',
 'total_home_epa',
 'total_away_epa',
 'total_home_rush_epa',
 'total_away_rush_epa',
 'total_home_pass_epa',
 'total_away_pass_epa',
 'air_epa',
 'yac_epa',
 'comp_air_epa',
 'comp_yac_epa',
 'total_home_comp_air_epa',
 'total_away_comp_air_epa',
 'total_home_comp_yac_epa',
 'total_away_comp_yac_epa',
 'total_home_raw_air_epa',
 'total_away_raw_air_epa',
 'total_home_raw_yac_epa',
 'total_away_raw_yac_epa',
 'wp',
 'def_wp',
 'home_wp',
 'away_wp',
 'wpa',
 'home_wp_post',
 'away_wp_post',
 'total_home_rush_wpa',
 'total_away_rush_wpa',
 'total_home_pass_wpa',
 'total_away_pass_wpa',
 'air_wpa',
 'yac_wpa',
 'comp_air_wpa',
 'comp_yac_wpa',
 'total_home_comp_air_wpa',
 'total_away_comp_air_wpa',
 'total_home_comp_yac_wpa',
 'total_away_comp_yac_wpa',
 'total_home_raw_air_wpa',
 'total_away_raw_air_wpa',
 'total_

In [21]:
outcomes = pd.read_csv("Play Outcomes & Events1.csv")
outcomes.head()

,ydstogo,ydsnet,desc,yards_gained,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,fourth_down_converted,...,penalty_player_id,penalty_player_name,penalty_yards,replay_or_challenge,replay_or_challenge_result,penalty_type,aborted_play,success,first_down,out_of_bounds
0,0.0,60.0,4-P.Dawson kicks 67 yards from CLE 30 to IND 3...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,0.0,NaN,NaN,0.0,1.0,0.0,0.0
1,10.0,60.0,(14:52) 18-P.Manning pass to 35-R.Williams to ...,9.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,0.0,NaN,NaN,0.0,1.0,0.0,0.0
2,1.0,60.0,(14:24) 32-E.James right guard to IND 47 for n...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,0.0
3,1.0,60.0,(13:37) 18-P.Manning pass to 81-M.Pollard to C...,18.0,0.0,1.0,0.0,1.0,0.0,0.0,...,NaN,NaN,NaN,0.0,NaN,NaN,0.0,1.0,1.0,0.0
4,10.0,60.0,(13:27) (Shotgun) 18-P.Manning pass incomplete...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,0.0


In [31]:
outcomes = outcomes.fillna(0)

In [36]:
sum_stats = []
not_summed = ["penalty_player_id", "penalty_player_name","replay_or_challenge", "replay_or_challenge_result", "penalty_type"]
for col in list(outcomes.columns):
    if col == 'ydstogo' or col == 'ydsnet' or col == 'desc' or col in not_summed:
        continue
    else:
        sum_stats.append(col)
sum_stats

['yards_gained',
 'first_down_rush',
 'first_down_pass',
 'first_down_penalty',
 'third_down_converted',
 'third_down_failed',
 'fourth_down_converted',
 'fourth_down_failed',
 'incomplete_pass',
 'interception',
 'fumble_forced',
 'fumble_not_forced',
 'fumble_out_of_bounds',
 'solo_tackle',
 'safety',
 'penalty',
 'tackled_for_loss',
 'fumble_lost',
 'qb_hit',
 'rush_attempt',
 'pass_attempt',
 'sack',
 'touchdown',
 'pass_touchdown',
 'rush_touchdown',
 'return_touchdown',
 'fumble',
 'complete_pass',
 'assist_tackle',
 'lateral_reception',
 'lateral_rush',
 'lateral_return',
 'lateral_recovery',
 'penalty_yards',
 'aborted_play',
 'success',
 'first_down',
 'out_of_bounds']

In [37]:
len(sum_stats)

38

In [34]:
coltspbp = coltspbp.fillna(0)

In [38]:
#passing
weekly = coltspbp.groupby(["passer_player_name", "week", "play_type"])[cols].mean().join

In [45]:
passers = coltspbp[coltspbp["passer_player_name"] != 0]

In [48]:
cols.append("ydstogo")

In [62]:
sum_stats[0] = 'passing_yards'

In [63]:
sum_stats

['passing_yards',
 'first_down_rush',
 'first_down_pass',
 'first_down_penalty',
 'third_down_converted',
 'third_down_failed',
 'fourth_down_converted',
 'fourth_down_failed',
 'incomplete_pass',
 'interception',
 'fumble_forced',
 'fumble_not_forced',
 'fumble_out_of_bounds',
 'solo_tackle',
 'safety',
 'penalty',
 'tackled_for_loss',
 'fumble_lost',
 'qb_hit',
 'rush_attempt',
 'pass_attempt',
 'sack',
 'touchdown',
 'pass_touchdown',
 'rush_touchdown',
 'return_touchdown',
 'fumble',
 'complete_pass',
 'assist_tackle',
 'lateral_reception',
 'lateral_rush',
 'lateral_return',
 'lateral_recovery',
 'penalty_yards',
 'aborted_play',
 'success',
 'first_down',
 'out_of_bounds']

In [64]:

weekly_stats = (
    passers.groupby(['passer_player_name',"play_type",'week', 'posteam'])[sum_stats].sum()
    .join(passers.groupby(['passer_player_name',"play_type",'week', 'posteam'])[cols].mean())
    .reset_index()
)

In [65]:
weekly_stats

,passer_player_name,play_type,week,posteam,passing_yards,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,...,cp,cpoe,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,ydstogo
0,A.Brooks,pass,4,NO,166.0,0.0,8.0,0.0,2.0,4.0,...,0.0,0.0,-0.752571,0.0,0.0,0.0,0.0,0.0,0.0,8.033334
1,B.Griese,pass,9,MIA,231.0,0.0,8.0,0.0,0.0,8.0,...,0.0,0.0,-0.324513,0.0,0.0,0.0,0.0,0.0,0.0,9.000000
2,B.Huard,pass,4,IND,13.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.285624,0.0,0.0,0.0,0.0,0.0,0.0,11.000000
3,B.Huard,pass,15,IND,9.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.180409,0.0,0.0,0.0,0.0,0.0,0.0,7.000000
4,B.Huard,pass,18,IND,17.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,-0.465953,0.0,0.0,0.0,0.0,0.0,0.0,8.800000
5,B.Johnson,pass,5,TB,318.0,0.0,14.0,0.0,4.0,6.0,...,0.0,0.0,0.537694,0.0,0.0,0.0,0.0,0.0,0.0,9.473684
6,B.Johnson,qb_spike,5,TB,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.061063,0.0,0.0,0.0,0.0,0.0,0.0,10.000000
7,B.Leftwich,pass,3,JAX,32.0,0.0,2.0,0.0,1.0,0.0,...,0.0,0.0,0.803080,0.0,0.0,0.0,0.0,0.0,0.0,8.000000
8,B.Leftwich,pass,10,JAX,179.0,0.0,8.0,0.0,4.0,3.0,...,0.0,0.0,0.356997,0.0,0.0,0.0,0.0,0.0,0.0,9.545455
9,B.Volek,pass,2,TEN,61.0,0.0,4.0,0.0,2.0,0.0,...,0.0,0.0,-0.926888,0.0,0.0,0.0,0.0,0.0,0.0,7.700000


In [66]:
INDpasser_stats = weekly_stats[weekly_stats["posteam"] == 'IND'].reset_index(drop=True)
opppasser_stats = weekly_stats[weekly_stats["posteam"] != 'IND'].reset_index(drop=True)

In [67]:
INDpasser_stats


,passer_player_name,play_type,week,posteam,passing_yards,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,...,cp,cpoe,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,ydstogo
0,B.Huard,pass,4,IND,13.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.285624,0.0,0.0,0.0,0.0,0.0,0.0,11.000000
1,B.Huard,pass,15,IND,9.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.180409,0.0,0.0,0.0,0.0,0.0,0.0,7.000000
2,B.Huard,pass,18,IND,17.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,-0.465953,0.0,0.0,0.0,0.0,0.0,0.0,8.800000
3,P. Manning,pass,19,IND,304.0,0.0,17.0,0.0,7.0,0.0,...,0.0,0.0,0.898036,0.0,0.0,0.0,0.0,0.0,0.0,7.870968
4,P. Manning,pass,20,IND,237.0,0.0,13.0,0.0,3.0,9.0,...,0.0,0.0,-0.281669,0.0,0.0,0.0,0.0,0.0,0.0,8.764706
5,P.Manning,pass,1,IND,211.0,0.0,14.0,0.0,6.0,4.0,...,0.0,0.0,-0.056372,0.0,0.0,0.0,0.0,0.0,0.0,8.113636
6,P.Manning,pass,2,IND,173.0,0.0,8.0,0.0,1.0,6.0,...,0.0,0.0,0.176662,0.0,0.0,0.0,0.0,0.0,0.0,9.272727
7,P.Manning,pass,3,IND,216.0,0.0,11.0,0.0,6.0,7.0,...,0.0,0.0,0.144375,0.0,0.0,0.0,0.0,0.0,0.0,8.121212
8,P.Manning,pass,4,IND,314.0,0.0,14.0,0.0,4.0,3.0,...,0.0,0.0,1.052818,0.0,0.0,0.0,0.0,0.0,0.0,8.692307
9,P.Manning,pass,5,IND,386.0,0.0,17.0,1.0,5.0,9.0,...,0.0,0.0,0.365930,0.0,0.0,0.0,0.0,0.0,0.0,9.458333


In [68]:
opppasser_stats

,passer_player_name,play_type,week,posteam,passing_yards,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,...,cp,cpoe,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,ydstogo
0,A.Brooks,pass,4,NO,166.0,0.0,8.0,0.0,2.0,4.0,...,0.0,0.0,-0.752571,0.0,0.0,0.0,0.0,0.0,0.0,8.033334
1,B.Griese,pass,9,MIA,231.0,0.0,8.0,0.0,0.0,8.0,...,0.0,0.0,-0.324513,0.0,0.0,0.0,0.0,0.0,0.0,9.000000
2,B.Johnson,pass,5,TB,318.0,0.0,14.0,0.0,4.0,6.0,...,0.0,0.0,0.537694,0.0,0.0,0.0,0.0,0.0,0.0,9.473684
3,B.Johnson,qb_spike,5,TB,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.061063,0.0,0.0,0.0,0.0,0.0,0.0,10.000000
4,B.Leftwich,pass,3,JAX,32.0,0.0,2.0,0.0,1.0,0.0,...,0.0,0.0,0.803080,0.0,0.0,0.0,0.0,0.0,0.0,8.000000
5,B.Leftwich,pass,10,JAX,179.0,0.0,8.0,0.0,4.0,3.0,...,0.0,0.0,0.356997,0.0,0.0,0.0,0.0,0.0,0.0,9.545455
6,B.Volek,pass,2,TEN,61.0,0.0,4.0,0.0,2.0,0.0,...,0.0,0.0,-0.926888,0.0,0.0,0.0,0.0,0.0,0.0,7.700000
7,B.Volek,qb_spike,2,TEN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.158042,0.0,0.0,0.0,0.0,0.0,0.0,10.000000
8,C.Hentrich,pass,2,TEN,10.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,2.710740,0.0,0.0,0.0,0.0,0.0,0.0,1.000000
9,C.Pennington,pass,11,NYJ,219.0,0.0,6.0,0.0,0.0,5.0,...,0.0,0.0,0.481661,0.0,0.0,0.0,0.0,0.0,0.0,10.352942


In [70]:
sum_stats2 = sum_stats
sum_stats2[0] = 'rushing_yards'
sum_stats2

['rushing_yards',
 'first_down_rush',
 'first_down_pass',
 'first_down_penalty',
 'third_down_converted',
 'third_down_failed',
 'fourth_down_converted',
 'fourth_down_failed',
 'incomplete_pass',
 'interception',
 'fumble_forced',
 'fumble_not_forced',
 'fumble_out_of_bounds',
 'solo_tackle',
 'safety',
 'penalty',
 'tackled_for_loss',
 'fumble_lost',
 'qb_hit',
 'rush_attempt',
 'pass_attempt',
 'sack',
 'touchdown',
 'pass_touchdown',
 'rush_touchdown',
 'return_touchdown',
 'fumble',
 'complete_pass',
 'assist_tackle',
 'lateral_reception',
 'lateral_rush',
 'lateral_return',
 'lateral_recovery',
 'penalty_yards',
 'aborted_play',
 'success',
 'first_down',
 'out_of_bounds']

In [73]:
rushers = coltspbp[coltspbp["rusher_player_name"] != 0]

In [88]:
#rushing
rushing_stats = (
    rushers.groupby(['rusher_player_name',"play_type",'week', 'posteam'])[sum_stats2].sum()
    .join(rushers.groupby(['rusher_player_name',"play_type",'week', 'posteam'])[cols].mean())
    .reset_index()
)

In [75]:
rushing_stats

,rusher_player_name,play_type,week,posteam,rushing_yards,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,...,cp,cpoe,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,ydstogo
0,A.Brooks,run,4,NO,9.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,-0.517141,0.0,0.0,0.0,0.0,0.0,0.0,9.000000
1,A.Smith,run,20,NE,100.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.124213,0.0,0.0,0.0,0.0,0.0,0.0,8.227273
2,A.Stecker,run,5,TB,22.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,-0.441475,0.0,0.0,0.0,0.0,0.0,0.0,11.555555
3,B.Huard,run,4,IND,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-1.061953,0.0,0.0,0.0,0.0,0.0,0.0,11.000000
4,B.Huard,run,15,IND,9.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.209059,0.0,0.0,0.0,0.0,0.0,0.0,12.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140,T.Jones,run,5,TB,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.401683,0.0,0.0,0.0,0.0,0.0,0.0,6.000000
141,T.Minor,run,9,MIA,11.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.815576,0.0,0.0,0.0,0.0,0.0,0.0,3.000000
142,T.Walters,run,9,IND,6.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.165376,0.0,0.0,0.0,0.0,0.0,0.0,10.000000
143,Team,qb_kneel,4,IND,-1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-3.051928,0.0,0.0,0.0,0.0,0.0,0.0,10.000000


In [78]:
INDrushing_stats = rushing_stats[rushing_stats["posteam"] == "IND"].reset_index(drop=True)
opprushing_stats = rushing_stats[rushing_stats["posteam"] != "IND"].reset_index(drop=True)

In [79]:
INDrushing_stats
opprushing_stats

,rusher_player_name,play_type,week,posteam,rushing_yards,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,...,cp,cpoe,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,ydstogo
0,A.Brooks,run,4,NO,9.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,-0.517141,0.0,0.0,0.0,0.0,0.0,0.0,9.000000
1,A.Smith,run,20,NE,100.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.124213,0.0,0.0,0.0,0.0,0.0,0.0,8.227273
2,A.Stecker,run,5,TB,22.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,-0.441475,0.0,0.0,0.0,0.0,0.0,0.0,11.555555
3,B.Johnson,run,5,TB,8.0,1.0,0.0,0.0,1.0,1.0,...,0.0,0.0,-0.322359,0.0,0.0,0.0,0.0,0.0,0.0,4.666667
4,B.Johnson,run,20,NE,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.277684,0.0,0.0,0.0,0.0,0.0,0.0,4.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,T.Duckett,run,15,ATL,74.0,4.0,0.0,0.0,0.0,1.0,...,0.0,0.0,-0.042657,0.0,0.0,0.0,0.0,0.0,0.0,8.055555
78,T.Henry,run,12,BUF,77.0,6.0,0.0,1.0,0.0,1.0,...,0.0,0.0,-0.015455,0.0,0.0,0.0,0.0,0.0,0.0,8.227273
79,T.Jones,run,5,TB,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.401683,0.0,0.0,0.0,0.0,0.0,0.0,6.000000
80,T.Minor,run,9,MIA,11.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.815576,0.0,0.0,0.0,0.0,0.0,0.0,3.000000


In [80]:
INDrushing_stats

,rusher_player_name,play_type,week,posteam,rushing_yards,first_down_rush,first_down_pass,first_down_penalty,third_down_converted,third_down_failed,...,cp,cpoe,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,ydstogo
0,B.Huard,run,4,IND,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-1.061953,0.0,0.0,0.0,0.0,0.0,0.0,11.000000
1,B.Huard,run,15,IND,9.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.209059,0.0,0.0,0.0,0.0,0.0,0.0,12.000000
2,D. Rhodes,run,19,IND,18.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,-0.197962,0.0,0.0,0.0,0.0,0.0,0.0,6.400000
3,D. Rhodes,run,20,IND,16.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.101144,0.0,0.0,0.0,0.0,0.0,0.0,10.000000
4,D.Rhodes,run,4,IND,36.0,2.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.214105,0.0,0.0,0.0,0.0,0.0,0.0,6.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,R.Williams,run,10,IND,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.472866,0.0,0.0,0.0,0.0,0.0,0.0,9.833333
59,R.Williams,run,15,IND,-2.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.162260,0.0,0.0,0.0,0.0,0.0,0.0,10.000000
60,R.Williams,run,18,IND,-1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.878992,0.0,0.0,0.0,0.0,0.0,0.0,10.000000
61,T.Walters,run,9,IND,6.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.165376,0.0,0.0,0.0,0.0,0.0,0.0,10.000000


In [84]:
#receivers
receivers = coltspbp[coltspbp["receiver_player_name"] != 0]

In [85]:
sum_stats3 = sum_stats
sum_stats3[0] = "receiving_yards"

In [87]:
receiving_stats = (
    receivers.groupby(['receiver_player_name',"play_type",'week', 'posteam'])[sum_stats3].sum()
    .join(rushers.groupby(['rusher_player_name',"play_type",'week', 'posteam'])[cols].mean())
    .reset_index()
)

In [90]:
INDreceiving_stats = receiving_stats[receiving_stats["posteam"] == "IND"].reset_index(drop=True)
oppreceiving_stats = receiving_stats[receiving_stats["posteam"] != "IND"].reset_index(drop=True)

In [ ]:
INDreceiving_stats.to_csv("INDreceiving_stats1.csv")
oppreceiving_stats.to_csv("oppreceiving_stats1.csv")
INDrushing_stats.to_csv("INDrushing_weekly_stats1.csv")
opprushing_stats.to_csv("opprushing_weekly_stats1.csv")
INDpasser_stats.to_csv("INDpassing_weekly_stats1.csv")
opppasser_stats.to_csv("opppassing_weeklystats1.csv")

In [3]:
import tensorflow as tf

TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [4]:
pip show tensorflow

Name: tensorflow
Version: 2.19.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /anaconda/envs/azureml_py38/lib/python3.10/site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, requests, setuptools, six, tensorboard, tensorflow-io-gcs-filesystem, termcolor, typing-extensions, wrapt
Required-by: tensorflow-text
Note: you may need to restart the kernel to use updated packages.


In [6]:
import tensorflow as tf, numpy as np
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [12]:
import numpy as np
print("NumPy:", np.__version__)

NumPy: 1.23.5


In [9]:
import sys
print(sys.executable)

/anaconda/envs/azureml_py38/bin/python


In [13]:
import sys, numpy
print("Python executable:", sys.executable)
print("NumPy location:", numpy.__file__)
print("NumPy version:", numpy.__version__)

Python executable: /anaconda/envs/azureml_py38/bin/python
NumPy location: /anaconda/envs/azureml_py38/lib/python3.10/site-packages/numpy/__init__.py
NumPy version: 1.23.5
